# Class 10: Multi-Agent Systems

**Tools:** n8n, Relevance AI (relevanceai.com -- free tier), any LLM chat


### Agenda:
1. A clear understanding of **what multi-agent systems are** -- an orchestrator that breaks tasks down and delegates to specialist sub-agents
2. The **team pattern** burned into your brain: manager, researcher, writer, reviewer -- each agent doing its piece
3. A working **2-agent pipeline in n8n**: Agent 1 researches, Agent 2 turns research into a polished article
4. Hands-on experience with **Relevance AI's visual multi-agent builder** -- connect and configure agents without code
5. A practical **decision framework** for when to use multi-agent vs single-agent systems

## Section 1 -- Intro: Why One Agent Isn't Enough (10 min)

> "So far, every agent we've built has been a solo operator -- one LLM brain making all the decisions, using all the tools, handling the entire task from start to finish. And for many tasks, that works great.
>
> But think about how real software teams work. You don't have one engineer who writes the code, reviews their own code, writes the tests, writes the docs, does the deployment, and monitors production. You have **specialists**. Someone's great at architecture. Someone else is meticulous at testing. Another person writes docs that humans actually read.
>
> Multi-agent systems apply the same principle. Instead of one overloaded agent trying to do everything, you build a **team of specialist agents**, each focused on what it does best, coordinated by a manager agent that breaks the task down and delegates.
>
> Today we build that team."



**Why this matters for SWEs:**
- Multi-agent is the dominant architecture pattern in production AI systems at companies like Canva, KPMG, and Autodesk
- Frameworks like CrewAI, AutoGen, and LangGraph have made multi-agent systems the standard approach for complex tasks
- No-code platforms like Relevance AI now let non-engineers build multi-agent teams -- understanding the pattern lets you guide, review, and extend what they build
- The single-agent vs multi-agent decision is one of the most common architecture calls you'll make as an AI-literate engineer

## Section 2 -- Concept in 5 Minutes: The Multi-Agent Pattern

> "Let me show you the architecture. One diagram, and you'll get it."

![image](https://d2beiqkhq929f0.cloudfront.net/public_assets/assets/000/193/698/original/1.jpeg?1777449386)


**The key insight:** Each sub-agent has its own **system prompt**, its own **tools**, and its own **focus area**. The researcher has web search tools and instructions to gather facts. The writer has no tools -- just writing expertise in its system prompt. The reviewer has a checklist and instructions to be critical.

**One-line analogy:**
- **Single agent** = a freelancer who does everything themselves
- **Multi-agent** = a small agency with specialists, run by a project manager

### The Team Pattern -- How It Actually Works

> "Let me trace through a real example so you see exactly what happens at each step."

**Task:** "Write a 500-word blog post about the state of Rust in backend development in 2026."

![image](https://d2beiqkhq929f0.cloudfront.net/public_assets/assets/000/193/699/original/2.png?1777449435)


**Beginner Note:** Each "agent" is just an LLM call with a different system prompt and (sometimes) different tools. There's no magical agent software -- it's system prompts and routing.

**Intermediate Note:** Notice the WRITER has no tools -- it only writes. The RESEARCHER has search tools. Specialisation means each agent gets only the capabilities it needs. This reduces errors and hallucinations.

**Advanced Extension:** This pattern maps directly to code frameworks: in CrewAI, each agent is an `Agent()` object with a `role`, `goal`, `backstory`, and `tools` list. In LangGraph, each agent is a node in a graph with edges defining the flow.

## Section 3 -- Tool Demo: Building a 2-Agent Pipeline in n8n (15 min)

> "Let's build the simplest possible multi-agent system: two agents in a pipeline. Agent 1 researches. Agent 2 writes. We'll use n8n because you already know it from Class 6, and you can literally see each agent as a separate node on the canvas."


### The Architecture on Canvas

![image](https://d2beiqkhq929f0.cloudfront.net/public_assets/assets/000/193/700/original/3.png?1777449485)


### Step-by-step demo (instructor screen-shares):

**Step 1 -- Create a new n8n workflow with a Chat Trigger.**

**Step 2 -- Add the first AI Agent node (Researcher).**
- Rename the node to `Researcher Agent`
- Connect a Chat Model (OpenAI GPT-4o or Gemini)
- Connect tools: Web Search (SerpAPI) and optionally Web Browse
- Set the System Message:

In [1]:
# AGENT 1 SYSTEM MESSAGE — Researcher

# You are a tech research specialist.
#
# When given a topic, you must:
# 1. Search the web for recent, high-quality information
# 2. Read at least 2-3 sources in detail
# 3. Extract key facts, statistics, and trends
# 4. Note the source URL for every fact
#
# OUTPUT FORMAT (strictly follow this):
# TOPIC: [the topic]
# KEY FINDINGS:
# 1. [finding] (source: [url])
# 2. [finding] (source: [url])
# ...
# TRENDS: [2-3 emerging trends]
# GAPS: [what you could NOT find reliable info on]
#
# Rules:
# - Only include facts you found in search results
# - Never make up statistics or facts
# - If information is thin, say so explicitly
# - Output research notes ONLY — do NOT write an article

**Step 3 -- Add the second AI Agent node (Writer).**
- Click **+** after the Researcher Agent and add another **AI Agent** node
- Rename to `Writer Agent`
- Connect a Chat Model (same or different from Agent 1)
- **No tools** -- the Writer only writes, it doesn't search
- Set the System Message:

In [2]:
# AGENT 2 SYSTEM MESSAGE — Writer

# You are a senior tech blog writer.
#
# You will receive research notes from a research specialist.
# Your job is to transform those notes into a polished blog post.
#
# WRITING GUIDELINES:
# - Target audience: senior software engineers
# - Tone: technical but accessible, conversational not academic
# - Structure: hook intro, 3-4 sections with subheadings, conclusion
# - Length: approximately 500 words
# - Include data and statistics from the research notes
# - Naturally weave in source attributions
#
# Rules:
# - Only use information from the research notes provided
# - Do NOT invent facts, stats, or quotes
# - If the research notes flag gaps, mention what is unknown
# - End with a forward-looking conclusion

**Step 4 -- Wire them together.**

The key step: connect the **output** of the Researcher Agent to the **input** of the Writer Agent. In n8n, the Researcher's response text automatically flows as the user message to the Writer.

**Step 5 -- Test the pipeline.**

Open the Chat panel and send:
```
Write a blog post about the current state of WebAssembly in backend development.
```

**What happens (watch the execution log):**

```
STEP 1: Chat Trigger passes the request to Researcher Agent
STEP 2: Researcher Agent searches the web (you see tool calls in the log)
STEP 3: Researcher Agent outputs structured research notes
STEP 4: Research notes flow into Writer Agent as input
STEP 5: Writer Agent produces a polished blog post
STEP 6: Blog post returned to chat
```

> **Instructor says:** "Look at the execution panel. You can see EXACTLY where the research ends and the writing begins. Two separate agents, two separate system prompts, two separate purposes. The researcher never tries to write a blog post. The writer never tries to search the web. That's the power of specialisation."

**What success looks like:** The final blog post is better than what a single agent would produce because:
- Research is thorough (dedicated agent with search tools)
- Writing is polished (dedicated agent with writing-optimised prompt)
- Neither agent is distracted by the other's job

## Section 4 -- Hands-On Usage (25 min)

### Part A: Build the 2-Agent Pipeline Yourself (15 min)

### Exercise 4.1 -- Replicate and Test the Pipeline

**Open [app.n8n.cloud](https://app.n8n.cloud/) and build the 2-agent workflow from the demo.**

Use the system messages from above (or customise them to your preference).

**Test with these prompts:**

1. `Write a blog post about how AI is changing code review practices in 2026.`

2. `Write a blog post comparing Rust vs Go for building microservices.`

3. `Write a blog post about the security implications of AI coding assistants.`

**For each test, check:**
- Does the Researcher actually search (check the execution log for tool calls)?
- Are the research notes structured and sourced?
- Does the Writer produce a different style than raw research notes?
- Is the final article better than what you'd get from a single-agent prompt?

**Beginner Note:** Take it step by step. Build Agent 1 first, test it alone. Then add Agent 2 and wire them together.

**Intermediate Note:** Try swapping the Writer's model (e.g., use GPT-4o for research but Gemini for writing) and see if the writing style changes.

### Exercise 4.2 -- Add a Third Agent: The Reviewer

**Extend your pipeline with a review step:**

```
[Chat Trigger] --> [Researcher] --> [Writer] --> [Reviewer] --> [Output]
```

**Reviewer Agent system message:**

```
You are a senior editor and fact-checker.

You will receive a blog post draft. Your job is to review it for:
1. Factual accuracy — flag any claims that seem unsupported
2. Clarity — flag any confusing sentences or jargon
3. Structure — is the intro engaging? Does the conclusion land?
4. Tone — is it appropriate for senior developers?

OUTPUT FORMAT:
VERDICT: APPROVED or NEEDS REVISION
SCORE: X/10
ISSUES: (numbered list, or "None")
REVISED ARTICLE: (if you made fixes, include the full revised version)
```

**Test it and compare:** Is the 3-agent output better than the 2-agent output? Check for specific improvements the Reviewer caught.

**Advanced Extension:** Make the Reviewer loop -- if it says "NEEDS REVISION," have n8n route the output back to the Writer with the feedback, then re-submit to the Reviewer. This creates a revision cycle. (Use n8n's IF node to check the verdict.)

### Part B: Relevance AI -- Visual Multi-Agent Builder (10 min)

> "n8n is powerful but you're still wiring nodes manually. Relevance AI takes a different approach -- it's an AI-first platform where you build agents and tools visually, then compose them into teams. Think of it as the 'Figma for multi-agent systems.'"

### What is Relevance AI?

Relevance AI is a no-code platform for building AI agents and multi-agent teams. You can:
- Create specialist agents with custom instructions, tools, and knowledge bases
- Compose agents into teams with defined roles and handoff rules
- Deploy agents to run autonomously on triggers or schedules
- Monitor agent activity and costs from a dashboard

![image](https://d2beiqkhq929f0.cloudfront.net/public_assets/assets/000/193/702/original/4.jpeg?1777449537)


**Pricing:** Free tier includes unlimited agents and tools, 200 actions/month -- more than enough for today's exercises.

### Exercise 4.3 -- Explore Relevance AI's Multi-Agent Builder

**Open [relevanceai.com](https://relevanceai.com/) and create a free account.**

**Step 1 -- Create your first agent.**
- Click **"Create Agent"** (or use a template)
- Give it a name: `Tech Researcher`
- Write instructions similar to your n8n Researcher system prompt
- Add the **Web Search** tool from the tool library

**Step 2 -- Create a second agent.**
- Name: `Blog Writer`
- Instructions: writing-focused system prompt (no search capabilities)
- No tools needed

**Step 3 -- Build a multi-agent team.**
- Navigate to the **Multi-Agent** section
- Create a team with your Researcher and Writer
- Define the flow: Researcher runs first, passes output to Writer
- Relevance AI handles the handoff and data passing automatically

**Step 4 -- Test it.**
- Send a task: `Research and write a blog post about edge computing trends in 2026`
- Watch the agents work in sequence in the activity log

**What success looks like:** You see both agents execute in sequence -- Researcher first, then Writer. The final output is a polished article grounded in real search results. You built this without writing a single line of code, and the agent handoff was configured visually.

**Beginner Note:** Relevance AI's UI is very guided. If you get stuck, their template gallery has pre-built multi-agent teams you can clone and modify.


## Section 5 -- Guided Exercise: When to Use Multi-Agent (20 min)

> "Multi-agent sounds amazing in demos. But here's the truth: **more agents does not always mean better results**. Every agent you add introduces latency, cost, potential failure points, and complexity. The skill isn't building multi-agent systems -- it's knowing **when** to build them."


### The Trade-Off Framework

![image](https://d2beiqkhq929f0.cloudfront.net/public_assets/assets/000/193/703/original/5.jpeg?1777449579)


## The costs of Multi-Agents

![image](https://d2beiqkhq929f0.cloudfront.net/public_assets/assets/000/193/707/original/7.jpeg?1777449907)

### Exercise -- Single or Multi? Classification Game

**For each scenario, decide: single agent or multi-agent? Justify your choice.**

In [3]:
# Classify each scenario. Fill in your answers.

scenarios = {
    "1. Summarise a single article into 3 bullet points": {
        "answer": "",  # SINGLE or MULTI?
        "reasoning": "",
    },

    "2. Research a topic, write a report, fact-check it, "
    "then format as a PDF": {
        "answer": "",
        "reasoning": "",
    },

    "3. Translate a document from English to Spanish": {
        "answer": "",
        "reasoning": "",
    },

    "4. Monitor competitor pricing daily, compare with our "
    "prices, draft adjustment recommendations, get manager "
    "approval, then update the database": {
        "answer": "",
        "reasoning": "",
    },

    "5. Generate unit tests for a Python function": {
        "answer": "",
        "reasoning": "",
    },

    "6. Triage incoming support tickets: classify urgency, "
    "draft a response, have a QA agent review the response, "
    "then send if approved or escalate if not": {
        "answer": "",
        "reasoning": "",
    },
}

# ANSWERS:
# 1. SINGLE — simple task, one skill (summarisation), no tools needed.
# 2. MULTI — 4 distinct phases, different skills per phase (research
#    needs search tools, writing needs no tools, fact-checking needs
#    search again, formatting needs file tools). Classic pipeline.
# 3. SINGLE — one task, one skill. Adding agents would add latency
#    with no quality gain.
# 4. MULTI — monitoring, analysis, writing, approval, and DB update
#    are fundamentally different capabilities. The approval step
#    requires human-in-the-loop, which is a natural agent boundary.
# 5. SINGLE — unless you want a separate agent to review the tests,
#    one well-prompted agent handles this fine.
# 6. MULTI — classification, drafting, and QA are distinct skills.
#    The QA agent provides a quality gate. The escalation path
#    requires a decision agent.

**Beginner Note:** Keep it to 2-3 agents. Every agent you add should have a clear reason to exist as a separate specialist.

**Advanced Extension:** Consider failure modes. What happens if Agent 2 produces bad output? Does Agent 3 (reviewer) loop back, or does it escalate to a human? Design the error path, not just the happy path.

## Section 6 -- Mini Build: Your Multi-Agent Content Pipeline (25 min)

> "Time to build something real that uses the multi-agent pattern. Pick a challenge and go."



### Challenge A: The Research-to-Article Pipeline (n8n)

**Build a 3-agent pipeline in n8n:**
1. **Researcher:** Searches the web, outputs structured notes with sources
2. **Writer:** Transforms notes into a polished 500-word article
3. **Headline Generator:** Takes the article and generates 5 headline options with a clickability score for each

Add a final step that saves the article + best headline to a Google Doc.

Test with: `Write a blog post about how small AI models are outperforming large ones for specific tasks.`



### Challenge B: The Code Quality Pipeline (n8n)

**Build a 2-agent pipeline:**
1. **Code Generator:** Takes a feature description and writes Python code
2. **Code Reviewer:** Reviews the code for bugs, security issues, and style -- outputs a verdict + improved code

Test with: `Write a FastAPI endpoint that accepts a JSON payload, validates it with Pydantic, stores it in SQLite, and returns the record ID.`

## Section 7 -- Wrap-Up (10 min)

### The Multi-Agent Mental Model

![image](https://d2beiqkhq929f0.cloudfront.net/public_assets/assets/000/193/704/original/6.png?1777449626)


### What we built and learned today:

| Skill | What You Can Do Now |
|:---|:---|
| **Multi-agent concept** | Explain what multi-agent systems are and how an orchestrator delegates to specialists |
| **Team pattern** | Design a manager-researcher-writer-reviewer pipeline for content tasks |
| **n8n multi-agent** | Build 2-3 agent pipelines in n8n where each agent has its own prompt and tools |
| **Relevance AI** | Create and compose agent teams visually in a no-code multi-agent builder |
| **Decision framework** | Choose single-agent vs multi-agent based on complexity, cost, latency, and quality trade-offs |



### Key takeaways

1. **Multi-agent = specialisation.** Each agent has one job, one system prompt, and only the tools it needs. The system's intelligence comes from coordination, not from making one agent smarter.

2. **The pipeline pattern is the workhorse.** Agent 1 does X, passes output to Agent 2 which does Y, passes to Agent 3 which does Z. Simple, predictable, debuggable. Start here.

3. **The review loop is the quality multiplier.** Adding a reviewer agent that can send work back for revision is the single biggest quality improvement in multi-agent systems.

4. **More agents is not always better.** Every agent adds latency, cost, and failure surface. The right question isn't "how many agents?" but "does this task have distinct phases that benefit from separate specialisation?"

5. **n8n and Relevance AI are complementary.** n8n excels when you need to mix AI and non-AI steps in complex workflows. Relevance AI excels when the workflow IS the agents -- creation, composition, and deployment of agent teams.

6. **This is how production AI is built in 2026.** Companies running AI at scale use multi-agent architectures -- not because it's trendy, but because specialisation produces better, more reliable results than monolithic agents.

